# Screen calibration from shared cells

Recover the true **pixel <-> azimuth** mapping for each eye's goggles screen, using the cells
themselves as the common reference frame.

The `distance` session sweeps a **real ball on an arm** through -30..+60 deg, so it measures each
cell's true angular position independently of any screen. The `goggles-monocular` session gives
the **pixel** each cell fired at. All sessions come from one suite2p split, so they share cell
ids: pair each cell's true azimuth `theta` with its pixel `p`, regress `p` on `theta`, and the
slope/intercept are that screen's real deg->pixel mapping — the four arguments of
`teensyStimulusAngle(left_slope, left_intercept, right_slope, right_intercept)`.

This is **not circular**. The stimulus fit only chose which pixels got swept; the pixel<->theta
correspondence comes from the neuron.

Session `260715_SAM045` — the only date with all three experiments recorded.

In [ ]:
import sys, os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base_dir = r"C:\Users\acostamendoza\projects\distance-project"
for p in (base_dir, base_dir + r"\disparity-analysis"):
    if p not in sys.path:
        sys.path.append(p)

from src.data.DisparityDevice import DisparityDevice
from src.comparison.CrossExperimentAnalysis import CrossExperimentAnalysis

DATA = r"D:\data-distance-coding-project\processed-data"
mouse, date = "SAM045", "260715"


def session_pkl(experiment):
    session = f"{date}_{mouse}_{experiment}"
    return os.path.join(DATA, session, f"{session}_dataframe.pkl")


# The calibration lives entirely in the per-cell dataframes, so load them directly
# rather than through RadialDevice: CrossExperimentAnalysis is deliberately
# pandas-only, and keeping radial-analysis off sys.path avoids the namespace
# collision between the two experiments' 'src' packages.
ana = CrossExperimentAnalysis.from_paths(
    session_pkl("distance"),            # ball on an arm -> each cell's true azimuth
    session_pkl("goggles-monocular"),   # one eye at a time -> the pixel it fired at
    session_pkl("goggles-disparity"),
)

# The disparity device is still wanted: it carries the stimulus' own deg->pixel
# fits and the sweep geometry used for the on-panel check below.
S = DisparityDevice(mouse_name=mouse, record_date=date,
                        experiment_name="goggles-disparity")

RHO = 0.8   # split-half correlation floor, applied to the distance session and to the eye

%load_ext autoreload
%autoreload 2

## 1. Provenance — what the session actually ran on

Two things to establish before fitting anything: the sessions really do share cells (otherwise
the join is meaningless), and the stimulus fits in the log are **placeholders**.

In [ ]:
# The three sessions are one suite2p run, split -- so cell ids correspond and the
# join is meaningful. Check that on the ids the merge actually uses.
ids = {
    "distance":  set(ana.radial_df["cell_id"].astype(int)),
    "monocular": set(ana.monocular_df["cell"].astype(int)),
    "disparity": set(ana.disparity_df["cell"].astype(int)),
}
print("identical cell id sets across all three:",
      ids["distance"] == ids["monocular"] == ids["disparity"])
print("cells per session:", {k: len(v) for k, v in ids.items()})

ana.print_summary()

print("\nDeg->pixel fits this session actually ran on (from the stim log):")
print(f"  left_fit  = {S.left_fit}")
print(f"  right_fit = {S.right_fit}")
print("\nSlope = 1 px/deg on both eyes: these are PLACEHOLDERS, not a calibration.")
print("So every 'degree' label on the goggles data is NOMINAL until refit below.")

## 2. Pixel vs true azimuth

Each point is a cell: x is its true azimuth from the ball, y is the pixel it fired at in the
goggles.

Both eyes are drawn on the stimulus' **signed joint axis** — left panel negative, right panel
positive, meeting at a nasal edge of 0. The monocular dataframe stores the left eye's *drawn
panel* pixel (`43 - deg`), which is the negation of its signed-axis pixel (`deg - 43`), so the
left eye is negated to put both eyes in `teensyStimulusAngle`'s convention.

In [ ]:
for eye in ("right", "left"):
    ana.plot_rf_scatter(eye=eye, corr_threshold=RHO)
    plt.show()

ana.plot_rf_scatter_both(corr_threshold=RHO)
plt.show()

## 3. The fits

`theta` is not known exactly — it is measured, and quantized to the sweep's per-frame step
(90 deg / 59 frames = 1.525 deg). Ordinary least squares assumes the x variable is exact, so it
**attenuates the slope toward zero**. Orthogonal distance regression (ODR) allows error in both
variables. The OLS->ODR gap is the size of that attenuation.

Read `r` first: a slope is meaningless if pixel and azimuth are not actually related.

In [ ]:
rows = []
for eye in ("left", "right"):
    for thresh in (None, RHO):
        for method in ("ols", "odr"):
            f = ana.fit_eye_calibration(eye, corr_threshold=thresh, method=method)
            rows.append({
                "eye": eye,
                "rho": "-" if thresh is None else thresh,
                "method": method,
                "n": f["n"],
                "r": round(f["r"], 3),
                "slope": round(f["slope"], 3),
                "intercept": round(f["intercept"], 2),
            })

fits = pd.DataFrame(rows)
fits

## 4. Sanity checks

From the handoff doc's checklist:

- right slope positive; left slope positive on the signed axis
- intercepts should land near the placeholders (-43, +62) if the rig geometry is roughly as
  assumed — the **slope** is the part expected to move
- feed the recovered numbers back through the stimulus arithmetic and confirm the swept pixels
  stay on-panel across -30..60 at every disparity

The left eye's per-disparity frame counts below (60/55/48/42/35) are a useful independent check:
they are pure geometry, and they reproduce the handoff doc's table exactly.

In [ ]:
fitL = ana.fit_eye_calibration("left",  corr_threshold=RHO, method="odr")
fitR = ana.fit_eye_calibration("right", corr_threshold=RHO, method="odr")

R_FLOOR = 0.3   # below this the pixel/azimuth relationship is not there at all

print("sign (signed-axis convention: both slopes should be > 0)")
print(f"  right slope {fitR['slope']:+.3f}  ->  {'OK' if fitR['slope'] > 0 else 'WRONG SIGN'}")
print(f"  left  slope {fitL['slope']:+.3f}  ->  {'OK' if fitL['slope'] > 0 else 'WRONG SIGN'}")

print("\nrecovered vs placeholder")
for name, f, placeholder in (("left", fitL, S.left_fit), ("right", fitR, S.right_fit)):
    print(f"  {name:5s} slope     {f['slope']:+7.3f}   placeholder {placeholder[0]:+.0f}")
    print(f"  {name:5s} intercept {f['intercept']:+7.2f}   placeholder {placeholder[1]:+.0f}")

print("\nis the fit real?  (|r| >= %.1f)" % R_FLOOR)
for name, f in (("left", fitL), ("right", fitR)):
    verdict = "USABLE" if abs(f["r"]) >= R_FLOOR else "NOT RECOVERABLE - no pixel/azimuth relationship"
    print(f"  {name:5s} r={f['r']:+.3f}  n={f['n']:3d}  ->  {verdict}")

In [ ]:
# Feed the recovered numbers back through the stimulus arithmetic. The left eye is not
# recoverable (see above), so only the right eye is swapped in; the left keeps its placeholder.
PANEL_W = 240   # each panel spans 240 px; signed axis: left [-240, 0], right [0, +240]


def on_panel(px, eye, width=PANEL_W):
    """A pixel is drawable when it is on that eye's half of the signed joint axis."""
    return (px >= -width) & (px <= 0) if eye == "left" else (px >= 0) & (px <= width)


saved_left, saved_right = S.left_fit.copy(), S.right_fit.copy()
S.right_fit = np.array([fitR["slope"], fitR["intercept"]])   # left deliberately left alone
try:
    print(f"left_fit  = {S.left_fit}   (placeholder, unchanged)")
    print(f"right_fit = {S.right_fit}   (recovered)\n")
    for d in S.disparities:
        for eye in ("left", "right"):
            px = S.get_px_range(eye, disparity=d)
            vis = on_panel(px, eye)
            # The left eye running off its panel is inherent, not a fault: disparity is
            # implemented as a left-eye angular shift against a finite sweep, so the
            # further the shift, the earlier the target leaves the panel and goes gray.
            note = "" if vis.all() else f"   ({len(px) - vis.sum()} frames blank)"
            print(f"  D={d:+3d}  {eye:5s}  px {px.min():+7.1f} .. {px.max():+7.1f}   "
                  f"{vis.sum():2d}/{len(px)} frames on-panel{note}")
        print()
finally:
    S.left_fit, S.right_fit = saved_left, saved_right   # never leave the device mutated

## 5. Deliverable

The four `teensyStimulusAngle` arguments.

In [ ]:
print("teensyStimulusAngle(left_slope, left_intercept, right_slope, right_intercept)\n")

if abs(fitR["r"]) >= R_FLOOR:
    print(f"  right : slope {fitR['slope']:.2f}, intercept {fitR['intercept']:.1f}"
          f"   (r={fitR['r']:.2f}, n={fitR['n']}) -- RECOVERED")
if abs(fitL["r"]) < R_FLOOR:
    print(f"  left  : r={fitL['r']:+.2f} over n={fitL['n']} cells -- NOT RECOVERABLE, "
          f"keeping placeholder {list(saved_left)}")

print(f"\n  teensyStimulusAngle({saved_left[0]:.2f}, {saved_left[1]:.1f}, "
      f"{fitR['slope']:.2f}, {fitR['intercept']:.1f})")
print("\n  ^ left args are the UNCHANGED placeholder, not a calibration.")

## 6. Where this leaves us

**The right eye calibrates.** ODR recovers roughly `px = 0.90*deg + 62.4` — the intercept lands
almost exactly on the +62 placeholder, and the slope moves from the placeholder's 1.0 to ~0.9,
which is the parameter the doc expected to move (`teensyStimulusAngle`'s own docstring example
uses 0.88 for the right eye). OLS gives ~0.71 instead: that gap is the errors-in-variables
attenuation, and it is why ODR is the number to trust here.

**The left eye does not calibrate at all.** Not "underpowered" — absent. Across the 43 cells
reliable in both the distance session and the left eye, `r = +0.04`; tightening to rho>=0.8
leaves 15 cells at `r = -0.17`. There is no pixel/azimuth relationship to fit, so no slope from
this data means anything, and the intercept near -48 is just the mean of a flat cloud.

That is consistent with the recording rather than surprising: the hemisphere is contra-dominated
(ODI mean +0.25, `left_mean_corr` mean 0.11 vs `right_mean_corr` 0.41), and only 45/1489 cells
are left-eye reliable at all. **But left is the eye that carries disparity**, so this blocks the
disparity zero-point.

Open, from the handoff doc — not resolved here:

1. **Left eye.** Relaxing the reliability bar cannot manufacture a correlation that is not there.
   The remaining options are a shared geometric model letting left borrow strength from right's
   351 cells, or an independent constraint from the disparity session (with a correct left fit,
   population preferred-disparity should centre on 0).
2. **Parallax.** Each eye views the ball from a different position, so eye-centred azimuth differs
   from arm angle by a term that grows at near distances, with **opposite sign L vs R**. Pooled
   `binocular['RF']` averages over distances, leaving a few-degree systematic in each intercept —
   which moves the disparity zero-point, the one number the experiment is about.
3. **`distance` column.** The stim log says `distances = [0 1 2 3 4]` but the dataframe column only
   holds `{0,1,2,3}`. Worth confirming what it means before using it.